<a href="https://colab.research.google.com/github/AntonDozhdikov/politpredict/blob/main/political_marl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Многоагентная модель социальной системы (MADDPG)

К статье А.В. Дождикова «Средний класс как источник валидационной выборки для машинной модели политической системы и основной компонент функции награды в её обучении» (журнал **«Полис. Политические исследования»**).

## Запуск в Google Colab

1. **Runtime → Change runtime type → GPU (T4)**.
2. Выполните установку зависимостей в первой ячейке ниже.
3. Выполните последовательно все ячейки.
4. По окончании в папке `outputs/` появятся:
   - `figures/fig1..fig9.png` — графики Ч/Б,
   - `tables/table1..table6.xlsx` — таблицы с двуязычными заголовками,
   - `logs/summary_metrics.json` — численная сводка.

**Ориентир по времени:** ~25–40 мин на T4 при `EPISODES_TRAIN=120`.


In [1]:
!pip install -q tqdm scipy openpyxl matplotlib pandas torch


## Загрузка скрипта political_marl.py
Либо загрузите файл `political_marl.py` через панель «Files» в Colab, либо выполните ячейку ниже целиком (она содержит весь код).

In [2]:
"""
================================================================================
Многоагентная модель социальной системы как обучающегося алгоритма (MADDPG+DRL)
================================================================================

К статье А.В. Дождикова «Средний класс как источник валидационной выборки
для машинной модели политической системы и основной компонент функции награды
в её обучении» (журнал «Полис. Политические исследования»).

КОНЦЕПЦИЯ:
  Социальная система моделируется как 9-агентная среда классической стратификации:
      1. Элита (top ~1%)
      2. Высший средний класс
      3. Старый средний класс
      4. Новый средний класс — профессионалы
      5. Новый средний класс — служащие
      6. Квалифицированный рабочий класс
      7. Неквалифицированный рабочий класс
      8. Прекариат
      9. Маргинальная прослойка

  Каждый агент имеет свою функцию награды (классовый интерес: рента, занятость,
  доход, доверие, мобильность). Среда — единая социальная система с экзогенными
  кризисами (внешними по отношению к моделируемой системе).

  Функция награды политической системы в целом — взвешенная сумма по среднему
  классу (формула 3 в статье). Валидационная функция (формула 2) измеряет,
  насколько прогнозы агентов о реакции МК совпадают с фактической реакцией.

КАЛИБРОВКА:
  6 стран: Швеция, Германия, Россия, Бразилия, Казахстан, США.
  Доли классов и макроряды откалиброваны по реальным данным
  (World Bank, ИС РАН, BLS, IBGE, Eurostat, Росстат, OECD, WID.world).

ПЕРИОДЫ:
  Обучение: 2000–2025 (26 шагов, шаг = 1 год).
  Прогноз : 2026–2050 (25 шагов).

КРИЗИСЫ:
  ≥ 30 кризисных событий — исторические (2000–2025, по реальным датам)
  + перспективные (2026–2050, по сценариям IPCC AR6, WEF GRR 2025, OECD).

ЗАПУСК В GOOGLE COLAB:
  1. Runtime → Change runtime type → GPU (T4).
  2. !pip install -q tqdm scipy openpyxl
  3. Загрузить файл political_marl.py, затем:
     !python political_marl.py
     или в ячейках — последовательное выполнение блоков.
  4. По окончании — в папке outputs/ будут figures/*.png, tables/*.xlsx,
     logs/summary_metrics.json.

ВРЕМЯ ВЫПОЛНЕНИЯ:
  ~25–40 минут на T4 при стандартных гиперпараметрах
  (EPISODES=120, T_TRAIN=26, BATCH=256).
================================================================================
"""

from __future__ import annotations
import os, sys, time, json, math, random, traceback                   # core
from dataclasses import dataclass, field, asdict                       # data classes
from typing import Dict, List, Tuple, Optional, Any                    # type hints

import numpy as np                                                      # числовые массивы
import pandas as pd                                                     # таблицы
import matplotlib.pyplot as plt                                         # графики
from matplotlib import rcParams                                         # настройка стиля
from matplotlib.ticker import MaxNLocator                               # ось целых чисел
from scipy.stats import pearsonr, spearmanr                             # корреляции

import torch                                                            # глубокое обучение
import torch.nn as nn                                                   # модули сети
import torch.nn.functional as F                                         # функции (loss, актив.)
from torch.optim import Adam                                            # оптимизатор

# Прогресс-бар: tqdm есть в Colab по умолчанию
try:
    from tqdm.auto import tqdm                                          # удобный прогресс
except ImportError:                                                     # fallback
    def tqdm(it, **k):                                                  # затычка без UI
        return it

# ======================================================================
#                 0. Расширенный traceback + воспроизводимость
# ======================================================================

def install_traceback_hook():
    """Полный traceback при ошибке (включая локальные переменные)."""
    def excepthook(exc_type, exc, tb):                                  # перехват исключений
        print("\n" + "="*80, file=sys.stderr)                           # разделитель
        print("ПОЛНЫЙ TRACEBACK", file=sys.stderr)                      # заголовок
        print("="*80, file=sys.stderr)                                  # разделитель
        traceback.print_exception(exc_type, exc, tb)                    # стандартный TB
        # дополнительно — локальные переменные в самом нижнем фрейме
        if tb is not None:                                              # есть TB
            while tb.tb_next is not None:                               # идём в самый низ
                tb = tb.tb_next                                         # к причине
            print("\nЛокальные переменные в месте ошибки:", file=sys.stderr)
            for k, v in list(tb.tb_frame.f_locals.items())[:30]:        # первые 30
                try:                                                    # безопасно
                    sv = repr(v)                                        # представление
                    if len(sv) > 200: sv = sv[:200] + "..."             # ограничим
                    print(f"  {k:25s} = {sv}", file=sys.stderr)         # печать
                except Exception:                                       # объект не выводим
                    print(f"  {k:25s} = <не выводится>", file=sys.stderr)
        print("="*80 + "\n", file=sys.stderr)                           # разделитель
    sys.excepthook = excepthook                                         # установка

install_traceback_hook()                                                # активируем

SEED = 20260528                                                          # глобальный сид
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)         # CPU воспр.
torch.cuda.manual_seed_all(SEED)                                         # GPU воспр.
torch.backends.cudnn.deterministic = True                                # детерминизм cuDNN
torch.backends.cudnn.benchmark = False                                   # без поиска бэнч.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")    # авто-выбор GPU
print(f"[init] PyTorch {torch.__version__}  device={DEVICE}  "
      f"cuda={torch.cuda.is_available()}")                               # сводка
if torch.cuda.is_available():                                            # GPU?
    print(f"[init] GPU = {torch.cuda.get_device_name(0)}, "
          f"mem = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ======================================================================
#                 1. Папки вывода + стиль Ч/Б (требование «Полиса»)
# ======================================================================

OUTDIR = "outputs"                                                       # корневая папка
FIGDIR = os.path.join(OUTDIR, "figures")                                 # графики
TABDIR = os.path.join(OUTDIR, "tables")                                  # таблицы
LOGDIR = os.path.join(OUTDIR, "logs")                                    # логи
for d in (OUTDIR, FIGDIR, TABDIR, LOGDIR):
    os.makedirs(d, exist_ok=True)                                        # создать

rcParams.update({                                                        # стиль для Полиса
    "font.family": "DejaVu Sans",                                        # есть в Colab
    "font.size": 11,                                                     # >= 11pt
    "axes.linewidth": 0.8,                                               # тонкие оси
    "axes.edgecolor": "black",                                           # ч/б
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": True,                                                   # сетка для чтения
    "grid.color": "0.85",
    "grid.linestyle": ":",
    "grid.linewidth": 0.5,
    "figure.dpi": 130,                                                   # экранное dpi
    "savefig.dpi": 300,                                                  # печатное dpi
    "savefig.bbox": "tight",                                             # обрезка
    "axes.prop_cycle": plt.cycler(color=                                 # ч/б градации
        ["0.0", "0.20", "0.35", "0.50", "0.62", "0.74"]),
})

LINESTYLES = ["-", "--", "-.", ":", (0,(3,1,1,1)), (0,(5,1))]            # 6 стилей
MARKERS    = ["o", "s", "^", "D", "v", "X"]                              # 6 маркеров
GRAYSCALE  = ["0.0", "0.20", "0.35", "0.50", "0.62", "0.74"]             # 6 серых
HATCHES    = ["", "///", "...", "xxx", "\\\\\\", "+++"]                  # 6 паттернов

# ======================================================================
#                 2. Спецификации классов, стран и кризисов
# ======================================================================

# ---------- 2.1 Социальные классы (9) ----------
CLASS_NAMES_RU = [
    "Элита",
    "Высший средний класс",
    "Старый средний класс",
    "Новый средний класс — профессионалы",
    "Новый средний класс — служащие",
    "Квалифицированный рабочий класс",
    "Неквалифицированный рабочий класс",
    "Прекариат",
    "Маргинальная прослойка",
]
CLASS_NAMES_EN = [
    "Elite",
    "Upper Middle Class",
    "Old Middle Class",
    "New Middle Class — Professionals",
    "New Middle Class — Clerical",
    "Skilled Working Class",
    "Unskilled Working Class",
    "Precariat",
    "Marginalised Stratum",
]
N_AGENTS = len(CLASS_NAMES_RU)                                           # 9

# Средний класс — индексы 1..4 (классы 2..5 по нумерации в статье)
MIDDLE_CLASS_IDX = [1, 2, 3, 4]                                          # 4 группы среднего класса

@dataclass
class SocialAgent:
    """Один социальный класс как агент в среде MADDPG."""
    idx: int                                                             # 0..8
    name_ru: str
    name_en: str
    interest_vec: np.ndarray                                             # 5-мерный вектор интересов (см. R_t)
    risk_aversion: float                                                 # неприятие риска ∈ [0,1]

# Вектор интересов (β-веса): [Δm — мобильность, Δw — благосостояние,
#                              Δe — занятость, Δτ — доверие, Δg — Джини (с минусом)]
# Чем выше — тем сильнее реакция класса на эту переменную.
CLASS_INTERESTS = np.array([
    # m,    w,    e,    τ,    g
    [0.2, 1.8, 0.2, 0.3, 0.2],   # Элита: благосостояние/рента; равенство для них «отрицательно»
    [0.6, 1.5, 0.6, 0.8, 0.4],   # Высший средний класс
    [0.7, 1.3, 0.9, 0.7, 0.5],   # Старый средний класс
    [1.5, 1.4, 1.0, 1.1, 0.9],   # Новый СК — профессионалы
    [1.4, 1.2, 1.1, 1.0, 1.0],   # Новый СК — служащие
    [0.9, 1.1, 1.5, 0.8, 1.0],   # Квалифицированный рабочий
    [0.7, 1.0, 1.4, 0.6, 1.1],   # Неквалифицированный рабочий
    [0.5, 0.8, 1.6, 0.5, 1.3],   # Прекариат
    [0.3, 0.7, 1.2, 0.4, 1.4],   # Маргиналы
], dtype=np.float32)

RISK_AVERSION = np.array(                                                # неприятие риска по классу
    [0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70], dtype=np.float32)

# β-вектор для агрегированной системной награды (формула 3 статьи)
BETA_SYSTEM = np.array([1.5, 1.2, 0.9, 1.1, 1.0], dtype=np.float32)
# Соответствие: [Δm, Δw, Δe, Δτ, −Δg] — последний с минусом будет применён в коде

# ---------- 2.2 Страны ----------
COUNTRIES_RU = ["Швеция", "Германия", "Россия", "Бразилия", "Казахстан", "США"]
COUNTRIES_EN = ["Sweden", "Germany", "Russia", "Brazil", "Kazakhstan", "USA"]
COUNTRY_CODES = ["SE", "DE", "RU", "BR", "KZ", "US"]
N_COUNTRIES = len(COUNTRIES_RU)                                          # 6

# Доли классов (≈2022), сумма = 100. Источник: calibration_data.md.
CLASS_SHARES = np.array([
    # Эл  ВСК  СтСК  НСК-п  НСК-с  Кв.раб  Некв.раб  Прек  Марг
    [ 1.0,  9.0,  7.0, 22.0, 20.0, 18.0, 12.0,  7.0,  4.0],   # SE
    [ 1.0,  9.0,  8.0, 21.0, 22.0, 18.0, 10.0,  7.0,  4.0],   # DE
    [ 1.5,  5.0,  5.5, 10.0, 10.0, 22.0, 20.0, 14.0, 12.0],   # RU
    [ 1.0,  5.5,  6.5, 10.0, 15.0, 22.0, 20.0, 12.0,  8.0],   # BR
    [ 1.5,  6.0,  8.0,  8.0,  7.0, 22.0, 20.0, 16.0, 11.5],   # KZ
    [ 1.0,  9.0,  6.0, 17.0, 17.0, 18.0, 20.0,  8.0,  4.0],   # US
], dtype=np.float32)
# Проверка нормировки
for i, c in enumerate(COUNTRIES_RU):
    s = CLASS_SHARES[i].sum()
    if abs(s - 100.0) > 0.01:
        raise ValueError(f"Сумма долей классов для {c} ≠ 100: {s}")

# Макроряды 2000–2024 (25 лет) — ВВП рост %, Джини, CPI %, безработица %, доверие %.
# Источник: calibration_data.md (World Bank, Pew, Левада, OECD, Gallup, IBGE, BLS).
YEARS_HIST = list(range(2000, 2025))                                     # 2000..2024 (25 точек)

MACRO = {                                                                # ключ: код страны
    "SE": {
        "gdp_growth": [4.6,1.4,2.3,1.9,4.2,2.8,4.7,3.2,-0.9,-4.3,5.8,3.2,-0.4,1.1,2.3,4.4,2.1,1.9,1.8,2.6,-1.9,5.2,1.3,-0.2,0.8],
        "gini":       [27.0,26.5,26.2,25.3,26.1,26.8,26.4,27.1,28.1,27.3,27.7,27.6,27.6,28.8,28.4,29.2,29.6,28.8,30.0,29.3,28.9,29.8,31.6,29.3,29.5],
        "cpi":        [0.9,2.4,2.2,1.9,0.4,0.5,1.4,2.2,3.4,-0.5,1.2,3.0,0.9,-0.04,-0.2,-0.05,1.0,1.8,2.0,1.8,0.5,2.2,8.4,8.5,2.8],
        "unemp":      [5.5,4.7,5.0,5.6,6.7,7.8,7.1,6.2,6.2,8.4,8.6,7.8,8.0,8.1,8.0,7.4,7.0,6.7,6.4,6.8,8.3,8.8,7.4,7.6,8.4],
        "trust":      [55,52,50,48,50,48,50,52,50,46,47,47,44,44,48,52,50,55,52,55,58,56,43,45,45],
    },
    "DE": {
        "gdp_growth": [2.9,1.6,-0.2,-0.5,1.2,0.9,3.9,2.9,0.9,-5.5,4.1,3.8,0.5,0.4,2.2,1.7,2.2,2.8,1.1,1.0,-4.1,3.9,1.8,-0.9,-0.5],
        "gini":       [28.7,29.9,29.9,29.9,30.1,31.6,31.0,31.2,30.6,30.4,30.2,30.4,30.9,31.4,30.8,31.4,31.3,31.3,31.9,32.2,32.5,33.6,33.7,33.5,33.0],
        "cpi":        [1.4,2.0,1.4,1.0,1.7,1.5,1.6,2.3,2.6,0.3,1.1,2.1,2.0,1.5,0.9,0.5,0.5,1.5,1.7,1.4,0.1,3.1,6.9,5.9,2.3],
        "unemp":      [7.9,7.8,8.5,9.8,10.7,11.2,10.3,8.7,7.5,7.9,7.0,6.0,5.4,5.3,5.0,4.6,4.1,3.8,3.4,3.2,3.9,3.6,3.1,3.1,3.4],
        "trust":      [42,40,37,34,32,28,34,38,38,36,37,40,37,38,41,45,50,55,55,53,67,55,61,40,30],
    },
    "RU": {
        "gdp_growth": [10.0,5.1,4.7,7.3,7.2,6.4,8.2,8.5,5.2,-7.8,4.5,4.3,4.0,1.8,0.7,-2.0,0.2,1.8,2.8,2.2,-2.7,5.9,-1.4,4.1,4.3],
        "gini":       [37.1,36.9,37.3,40.0,40.3,41.3,41.0,42.3,41.6,39.8,39.5,39.7,40.7,40.9,36.9,36.5,36.7,35.5,35.3,35.7,33.7,35.1,33.9,33.0,33.0],
        "cpi":        [20.8,21.5,15.8,13.7,10.9,12.7,9.7,9.0,14.1,11.6,6.8,8.4,5.1,6.8,7.8,15.5,7.0,3.7,2.9,4.5,3.4,6.7,13.7,5.9,8.4],
        "unemp":      [10.6,9.0,7.9,8.2,7.8,7.1,7.1,6.0,6.2,8.3,7.4,6.6,5.5,5.5,5.2,5.6,5.6,5.2,4.9,4.5,5.6,4.7,3.9,3.1,2.4],
        "trust":      [38,40,42,43,44,46,48,50,54,44,44,42,46,43,48,52,54,52,50,44,49,50,74,74,74],
    },
    "BR": {
        "gdp_growth": [4.4,1.4,3.1,1.1,5.8,3.2,4.0,6.1,5.1,-0.1,7.5,4.0,1.9,3.0,0.5,-3.5,-3.3,1.3,1.8,1.2,-3.3,4.8,3.0,3.2,3.4],
        "gini":       [58.0,58.4,58.1,57.6,56.5,56.3,55.6,54.9,54.0,53.7,53.0,52.9,53.4,52.7,52.1,51.9,53.4,53.3,53.9,53.5,48.8,52.9,51.9,51.5,50.3],
        "cpi":        [7.0,6.8,8.5,14.7,6.6,6.9,4.2,3.6,5.7,4.9,5.0,6.6,5.4,6.2,6.3,9.0,8.7,3.4,3.7,3.7,3.2,8.3,9.3,4.6,4.4],
        "unemp":      [10.9,10.6,10.6,11.2,10.1,10.6,9.7,9.3,8.3,9.4,8.4,7.6,7.3,7.1,6.8,8.5,11.6,12.8,12.3,11.9,13.7,13.2,9.2,7.9,6.8],
        "trust":      [36,33,30,28,30,28,32,37,36,38,45,44,40,37,30,18,15,18,20,23,28,25,30,35,35],
    },
    "KZ": {
        "gdp_growth": [9.8,13.5,9.8,9.3,9.6,9.7,10.7,8.9,3.3,1.2,7.3,7.4,4.8,6.0,4.2,1.2,1.1,4.1,4.1,4.5,-2.5,4.3,3.2,5.1,5.0],
        "gini":       [36.0,36.0,34.8,33.7,31.8,30.0,30.2,30.1,28.5,28.2,28.0,28.0,28.2,27.1,27.0,26.8,27.2,27.5,27.8,28.0,28.7,29.2,29.0,29.0,29.5],
        "cpi":        [13.2,8.4,5.8,6.4,6.9,7.6,8.7,10.8,17.1,7.3,7.4,8.5,5.2,5.9,6.8,6.7,14.4,7.4,6.2,5.3,6.7,8.0,15.0,14.5,8.7],
        "unemp":      [12.8,10.4,9.3,8.8,8.4,8.1,7.8,7.3,6.6,6.6,5.8,5.4,5.3,5.2,5.1,4.9,5.0,4.9,4.9,4.8,4.9,5.6,4.9,4.8,4.8],
        "trust":      [50,52,54,55,56,57,58,58,60,55,57,60,62,62,58,52,50,53,55,58,62,58,52,55,57],
    },
    "US": {
        "gdp_growth": [4.1,1.0,1.7,2.8,3.8,3.5,2.8,2.0,0.1,-2.6,2.7,1.6,2.3,2.1,2.5,2.9,1.8,2.5,3.0,2.6,-2.2,6.1,2.5,2.9,2.8],
        "gini":       [40.1,40.6,40.4,40.8,40.5,41.3,41.7,41.1,41.1,40.9,40.2,41.2,41.2,40.9,41.7,41.5,41.3,41.4,41.8,41.9,40.0,39.7,41.7,41.8,41.8],
        "cpi":        [3.4,2.8,1.6,2.3,2.7,3.4,3.2,2.9,3.8,-0.4,1.6,3.2,2.1,1.5,1.6,0.1,1.3,2.1,2.4,1.8,1.2,4.7,8.0,4.1,2.9],
        "unemp":      [4.0,4.7,5.8,6.0,5.5,5.1,4.6,4.6,5.8,9.3,9.6,8.9,8.1,7.4,6.2,5.3,4.9,4.4,3.9,3.7,8.1,5.3,3.7,3.6,4.0],
        "trust":      [44,56,52,51,44,41,37,35,37,30,23,19,23,19,20,21,20,20,20,20,27,24,20,16,17],
    },
}

# Согласованность длин рядов
for code, d in MACRO.items():
    for k, arr in d.items():
        if len(arr) != len(YEARS_HIST):
            raise ValueError(f"{code}/{k}: длина {len(arr)} ≠ {len(YEARS_HIST)}")

# ---------- 2.3 Каталог кризисов ----------
@dataclass
class Crisis:
    """Кризисное событие — экзогенный шок к среде."""
    name_ru: str
    name_en: str
    year_start: int                                                      # год начала
    duration: int                                                        # продолжительность (лет)
    ctype: str                                                           # тип ("эконом", "клим", "техн", ...)
    countries: List[str]                                                 # коды затронутых стран
    severity: float                                                      # амплитуда [0..1]
    sign: Dict[str, float]                                               # знак воздействия по каналам:
                                                                          # gdp, cpi, unemp, trust, gini, mobility
    source: str                                                          # цитата источника

# --- 2.3.1 Исторические кризисы 2000–2025 (около 24 событий) ---
HIST_CRISES: List[Crisis] = [
    Crisis("Лопание пузыря dot-com и теракты 11.09",
           "Dot-com bust & 9/11", 2001, 2, "финансовый",
           ["US","DE","SE"], 0.5,
           {"gdp":-1, "cpi":-0.3, "unemp":+1, "trust":+0.5, "gini":+0.2, "mobility":-0.4},
           "NBER; Federal Reserve"),
    Crisis("Рецессия и реформы Hartz",
           "German recession & Hartz reforms", 2002, 3, "структурный",
           ["DE"], 0.45,
           {"gdp":-0.8, "cpi":-0.1, "unemp":+1.2, "trust":-0.6, "gini":+0.4, "mobility":-0.3},
           "Bundesagentur für Arbeit"),
    Crisis("Финансовый кризис и девальвация реала",
           "Brazilian real devaluation crisis", 2002, 2, "финансовый",
           ["BR"], 0.55,
           {"gdp":-0.5, "cpi":+1.5, "unemp":+0.6, "trust":-0.5, "gini":+0.3, "mobility":-0.4},
           "Banco Central do Brasil"),
    Crisis("Глобальный финансовый кризис",
           "Global Financial Crisis", 2008, 3, "финансовый",
           ["US","DE","SE","RU","BR","KZ"], 0.85,
           {"gdp":-1.5, "cpi":-0.4, "unemp":+1.4, "trust":-0.8, "gini":+0.5, "mobility":-0.7},
           "World Bank; NBER"),
    Crisis("Банковский кризис в Казахстане (BTA, Казкоммерцбанк)",
           "Kazakhstan banking crisis", 2009, 2, "финансовый",
           ["KZ"], 0.55,
           {"gdp":-0.8, "cpi":+0.5, "unemp":+0.5, "trust":-0.5, "gini":+0.2, "mobility":-0.3},
           "АФК РК; НБ РК"),
    Crisis("Долговой кризис еврозоны",
           "European debt crisis", 2011, 3, "финансовый",
           ["DE","SE"], 0.45,
           {"gdp":-0.7, "cpi":-0.2, "unemp":+0.5, "trust":-0.6, "gini":+0.3, "mobility":-0.3},
           "ECB; Eurostat"),
    Crisis("Внешние торгово-финансовые ограничения и нефтяной шок",
           "External trade and financial restrictions; oil shock", 2014, 2, "геоэкономический",
           ["RU"], 0.75,
           {"gdp":-1.2, "cpi":+1.8, "unemp":+0.3, "trust":+0.3, "gini":-0.2, "mobility":-0.5},
           "Банк России; Росстат"),
    Crisis("Двойная девальвация тенге и нефтяной шок",
           "Tenge double devaluation; oil price shock", 2014, 3, "сырьевой",
           ["KZ"], 0.6,
           {"gdp":-0.9, "cpi":+1.6, "unemp":+0.3, "trust":-0.4, "gini":+0.2, "mobility":-0.4},
           "НБ РК; МВФ"),
    Crisis("Бразильская рецессия и политический кризис",
           "Brazilian recession & political crisis (Lava Jato)", 2014, 3, "политэкономический",
           ["BR"], 0.8,
           {"gdp":-1.4, "cpi":+1.0, "unemp":+1.5, "trust":-1.2, "gini":+0.3, "mobility":-0.6},
           "IBGE; FGV IBRE"),
    Crisis("Пандемия COVID-19",
           "COVID-19 pandemic", 2020, 2, "биологический",
           ["US","DE","SE","RU","BR","KZ"], 0.9,
           {"gdp":-1.5, "cpi":+0.3, "unemp":+1.2, "trust":+0.4, "gini":-0.3, "mobility":-0.8},
           "WHO; World Bank"),
    Crisis("Январские события и нарушение логистических цепочек",
           "January events and supply-chain disruption", 2022, 1, "социально-политический",
           ["KZ"], 0.55,
           {"gdp":-0.4, "cpi":+1.0, "unemp":+0.3, "trust":-0.4, "gini":+0.2, "mobility":-0.4},
           "BTI 2024; World Bank"),
    Crisis("Новый раунд внешних торгово-финансовых ограничений",
           "New round of external trade-and-financial restrictions", 2022, 4, "геоэкономический",
           ["RU"], 0.8,
           {"gdp":-1.0, "cpi":+1.6, "unemp":-0.3, "trust":+0.6, "gini":-0.3, "mobility":-0.5},
           "ЦБ РФ; IMF Country Report 2023"),
    Crisis("Энергетический кризис в Европе",
           "European energy crisis", 2022, 3, "энергетический",
           ["DE","SE"], 0.7,
           {"gdp":-1.0, "cpi":+1.8, "unemp":+0.2, "trust":-0.7, "gini":+0.3, "mobility":-0.4},
           "IEA; Eurostat"),
    Crisis("Постпандемический инфляционный шок",
           "Post-pandemic inflation shock", 2022, 2, "финансовый",
           ["US","DE","SE","BR","KZ"], 0.65,
           {"gdp":-0.4, "cpi":+1.9, "unemp":-0.2, "trust":-0.6, "gini":+0.3, "mobility":-0.3},
           "BLS; Fed; ECB"),
    Crisis("Кризис цепочек поставок (полупроводники, логистика)",
           "Supply-chain crisis (chips & logistics)", 2021, 2, "логистический",
           ["DE","SE","US"], 0.5,
           {"gdp":-0.5, "cpi":+0.9, "unemp":+0.1, "trust":-0.3, "gini":+0.1, "mobility":-0.2},
           "McKinsey GI; OECD"),
    Crisis("Импичмент и политический кризис в Бразилии",
           "Brazil impeachment crisis", 2016, 2, "политический",
           ["BR"], 0.55,
           {"gdp":-0.6, "cpi":+0.3, "unemp":+0.7, "trust":-1.0, "gini":+0.2, "mobility":-0.5},
           "IBGE; IPEA"),
    Crisis("Кризис доверия в США (постипотечный)",
           "US post-mortgage trust crisis", 2010, 4, "социально-политический",
           ["US"], 0.5,
           {"gdp":-0.3, "cpi":0.0, "unemp":+0.5, "trust":-1.0, "gini":+0.3, "mobility":-0.4},
           "Pew Research Center"),
    Crisis("Скачок инфляции в Швеции и зоне евро",
           "Swedish and euro-zone inflation surge", 2023, 2, "финансовый",
           ["SE","DE"], 0.5,
           {"gdp":-0.5, "cpi":+1.4, "unemp":+0.2, "trust":-0.4, "gini":+0.2, "mobility":-0.2},
           "Riksbank; ECB"),
    Crisis("Замедление мирового спроса и торговая фрагментация",
           "Slowing global demand & trade fragmentation", 2023, 3, "геоэкономический",
           ["DE","SE","KZ","BR"], 0.45,
           {"gdp":-0.5, "cpi":+0.4, "unemp":+0.3, "trust":-0.3, "gini":+0.2, "mobility":-0.3},
           "WEF GRR 2025"),
    Crisis("Демографический сдвиг (старение, сокращение рабочей силы)",
           "Demographic shift (aging workforce)", 2015, 11, "демографический",
           ["DE","SE","RU"], 0.4,
           {"gdp":-0.3, "cpi":+0.1, "unemp":+0.1, "trust":-0.2, "gini":+0.2, "mobility":-0.3},
           "OECD; Eurostat"),
    Crisis("Рост неравенства в США и Латинской Америке",
           "Rising US and LatAm inequality", 2015, 10, "социальный",
           ["US","BR"], 0.45,
           {"gdp":-0.2, "cpi":+0.1, "unemp":+0.2, "trust":-0.6, "gini":+0.5, "mobility":-0.5},
           "Oxfam; WID.world"),
    Crisis("Климатические шоки 2010-х (засухи, наводнения)",
           "Climate shocks of the 2010s", 2010, 15, "климатический",
           ["BR","US","KZ","RU"], 0.4,
           {"gdp":-0.3, "cpi":+0.3, "unemp":+0.2, "trust":-0.2, "gini":+0.2, "mobility":-0.3},
           "IPCC AR5/AR6"),
    Crisis("Цифровая трансформация и автоматизация занятости",
           "Digital transformation & job automation", 2018, 7, "технологический",
           ["US","DE","SE","RU","KZ","BR"], 0.4,
           {"gdp":+0.2, "cpi":+0.1, "unemp":+0.4, "trust":-0.3, "gini":+0.3, "mobility":+0.3},
           "WEF Future of Jobs; McKinsey GI"),
    Crisis("Миграционный кризис в Европе",
           "European migration crisis", 2015, 4, "социально-политический",
           ["DE","SE"], 0.5,
           {"gdp":+0.1, "cpi":+0.2, "unemp":+0.2, "trust":-0.7, "gini":+0.2, "mobility":-0.2},
           "UNHCR; Eurostat"),
]

# --- 2.3.2 Перспективные кризисы 2026–2050 (20 событий, типовые) ---
PROSP_CRISES: List[Crisis] = [
    Crisis("Климатический шок — экстремальная жара и засухи",
           "Climate shock — extreme heat & droughts", 2028, 4, "климатический",
           ["KZ","BR","RU"], 0.6,
           {"gdp":-0.9, "cpi":+1.0, "unemp":+0.4, "trust":-0.4, "gini":+0.4, "mobility":-0.4},
           "IPCC AR6 SYR (2023)"),
    Crisis("Климатический шок — наводнения и подъём уровня океана",
           "Climate shock — floods & sea-level rise", 2032, 8, "климатический",
           ["US","BR"], 0.55,
           {"gdp":-0.7, "cpi":+0.6, "unemp":+0.3, "trust":-0.3, "gini":+0.3, "mobility":-0.3},
           "IPCC AR6; World Bank 2024"),
    Crisis("Массовая автоматизация и ИИ-замещение труда",
           "Mass automation & AI-driven labor displacement", 2030, 10, "технологический",
           ["US","DE","SE"], 0.55,
           {"gdp":+0.3, "cpi":+0.2, "unemp":+1.0, "trust":-0.5, "gini":+0.6, "mobility":+0.4},
           "McKinsey GI (2023); WEF Future of Jobs 2025"),
    Crisis("ИИ-пузырь и технологический финансовый кризис",
           "AI bubble burst & tech financial crisis", 2029, 2, "финансовый",
           ["US","DE","SE"], 0.6,
           {"gdp":-1.0, "cpi":-0.2, "unemp":+0.6, "trust":-0.5, "gini":+0.3, "mobility":-0.5},
           "WEF GRR 2025"),
    Crisis("Суперстарение и пенсионный кризис",
           "Super-aging & pension/budget crisis", 2035, 15, "демографический",
           ["DE","SE","RU"], 0.5,
           {"gdp":-0.5, "cpi":+0.3, "unemp":-0.2, "trust":-0.4, "gini":+0.3, "mobility":-0.4},
           "OECD Long-run Scenarios 2025"),
    Crisis("Новая пандемия (биоугроза)",
           "New pandemic (biothreat)", 2033, 3, "биологический",
           ["US","DE","SE","RU","BR","KZ"], 0.75,
           {"gdp":-1.2, "cpi":+0.3, "unemp":+1.0, "trust":+0.2, "gini":-0.2, "mobility":-0.7},
           "WEF GRR 2025; WHO"),
    Crisis("Энергопереход и дестабилизация сырьевого экспорта",
           "Energy transition & destabilisation of resource exports", 2032, 12, "энергетический",
           ["RU","KZ"], 0.65,
           {"gdp":-1.0, "cpi":+0.5, "unemp":+0.7, "trust":-0.5, "gini":+0.4, "mobility":-0.5},
           "OECD; IEA NZE 2050"),
    Crisis("Продовольственный кризис",
           "Food crisis", 2031, 3, "климатический-логистический",
           ["KZ","BR","RU"], 0.55,
           {"gdp":-0.5, "cpi":+1.4, "unemp":+0.4, "trust":-0.4, "gini":+0.4, "mobility":-0.3},
           "WEF GRR 2025; FAO"),
    Crisis("Крупный киберудар по критической инфраструктуре",
           "Major cyberattack on critical infrastructure", 2028, 1, "технологический",
           ["US","DE","SE","RU","BR","KZ"], 0.55,
           {"gdp":-0.5, "cpi":+0.3, "unemp":+0.2, "trust":-0.6, "gini":+0.2, "mobility":-0.3},
           "WEF GRR 2025; Lloyd's"),
    Crisis("Фрагментация мировой экономики (деглобализация)",
           "Trade fragmentation & deglobalisation", 2027, 10, "геоэкономический",
           ["DE","SE","KZ","BR"], 0.5,
           {"gdp":-0.6, "cpi":+0.4, "unemp":+0.3, "trust":-0.3, "gini":+0.3, "mobility":-0.4},
           "WEF GRR 2025; OECD"),
    Crisis("Системный кризис цепочек поставок",
           "Systemic supply-chain crisis", 2034, 3, "логистический",
           ["DE","US","SE"], 0.45,
           {"gdp":-0.5, "cpi":+0.8, "unemp":+0.2, "trust":-0.3, "gini":+0.1, "mobility":-0.2},
           "McKinsey GI; OECD"),
    Crisis("Дефицит критических ресурсов (вода, литий, РЗМ)",
           "Critical-resource scarcity (water, lithium, REE)", 2036, 14, "ресурсный",
           ["KZ","BR","US"], 0.5,
           {"gdp":-0.5, "cpi":+0.9, "unemp":+0.3, "trust":-0.3, "gini":+0.3, "mobility":-0.3},
           "WEF GRR 2025; IEA Critical Minerals 2023"),
    Crisis("Резкое снижение рождаемости — демографический переход",
           "Sharp fertility decline — demographic transition", 2030, 20, "демографический",
           ["DE","SE","RU","BR"], 0.45,
           {"gdp":-0.4, "cpi":+0.1, "unemp":-0.1, "trust":-0.2, "gini":+0.2, "mobility":-0.3},
           "UN DESA; Eurostat"),
    Crisis("Рост неравенства до критического порога",
           "Inequality reaching critical threshold", 2030, 15, "социальный",
           ["US","BR","RU"], 0.55,
           {"gdp":-0.4, "cpi":+0.1, "unemp":+0.3, "trust":-0.8, "gini":+0.7, "mobility":-0.6},
           "WEF GRR 2025; Oxfam"),
    Crisis("Легитимационный и институциональный кризис",
           "Legitimacy & institutional crisis", 2028, 8, "политический",
           ["US","DE","BR"], 0.5,
           {"gdp":-0.3, "cpi":+0.1, "unemp":+0.2, "trust":-1.0, "gini":+0.3, "mobility":-0.4},
           "Gallup 2025; Pew 2024; Edelman Trust 2023"),
    Crisis("Массовая миграция — кризис приёма",
           "Mass migration — host-country crisis", 2035, 10, "демографически-социальный",
           ["DE","SE","US"], 0.5,
           {"gdp":+0.0, "cpi":+0.3, "unemp":+0.3, "trust":-0.7, "gini":+0.3, "mobility":-0.2},
           "World Bank Groundswell 2021; UNHCR"),
    Crisis("Финтех/крипто пузырь и финансовая нестабильность",
           "FinTech/crypto bubble & financial instability", 2034, 2, "финансовый",
           ["US","DE"], 0.45,
           {"gdp":-0.4, "cpi":+0.1, "unemp":+0.3, "trust":-0.4, "gini":+0.3, "mobility":-0.3},
           "BIS; FSB"),
    Crisis("Биотехнологические риски и синтетические патогены",
           "Biotech risks & synthetic pathogens", 2040, 3, "биологический-технологический",
           ["US","DE","SE","RU","BR","KZ"], 0.6,
           {"gdp":-0.9, "cpi":+0.4, "unemp":+0.7, "trust":-0.3, "gini":+0.2, "mobility":-0.5},
           "WEF GRR 2025; NTI"),
    Crisis("Хроническое загрязнение окружающей среды",
           "Chronic environmental pollution", 2030, 20, "экологический",
           ["BR","KZ","US","DE","SE","RU"], 0.4,
           {"gdp":-0.3, "cpi":+0.2, "unemp":+0.1, "trust":-0.3, "gini":+0.1, "mobility":-0.2},
           "OECD Environmental Outlook 2050"),
    Crisis("Коллапс биоразнообразия и экосистемных услуг",
           "Biodiversity collapse & ecosystem-services loss", 2038, 12, "экологический",
           ["BR","KZ","SE"], 0.5,
           {"gdp":-0.5, "cpi":+0.5, "unemp":+0.3, "trust":-0.3, "gini":+0.3, "mobility":-0.4},
           "WEF GRR 2025; IPBES 2019"),
]

ALL_CRISES = HIST_CRISES + PROSP_CRISES
print(f"[init] Всего кризисов в каталоге: {len(ALL_CRISES)} "
      f"(исторических {len(HIST_CRISES)}, перспективных {len(PROSP_CRISES)})")
assert len(ALL_CRISES) >= 30, "Требование: не менее 30 кризисов"

# ======================================================================
#                 3. Среда: социальная система на 1-летнем шаге
# ======================================================================

# Размерности
STATE_DIM = 12     # [gdp, cpi, unemp, trust, gini, m, w, e, τ_avg, crisis_intensity, year_norm, country_id]
ACTION_DIM = 5     # [Δm, Δw, Δe, Δτ, Δg-correction] — управляющие сигналы агента
GLOBAL_STATE_DIM = STATE_DIM + N_AGENTS * ACTION_DIM  # для critic: state + actions

@dataclass
class CountryState:
    """Текущее состояние одной страны в одном году."""
    code: str                                                            # SE / DE / RU / ...
    year: int                                                            # текущий год
    gdp: float                                                           # ВВП рост, %
    cpi: float                                                           # инфляция, %
    unemp: float                                                         # безработица, %
    trust: float                                                         # доверие, %
    gini: float                                                          # коэф. Джини
    mobility: float                                                      # социальная мобильность 0..1
    wealth: float                                                        # благосостояние 0..1
    employment: float                                                    # занятость 0..1
    tau_avg: float                                                       # средний уровень доверия (отдельный канал)
    crisis_intensity: float                                              # 0..1 — суммарная сила кризисов в этом году
    class_shares: np.ndarray                                             # текущие доли классов (9,)

    def as_state_vector(self, country_idx: int) -> np.ndarray:
        """Преобразовать в вектор состояния (STATE_DIM,) для нейросетей."""
        return np.array([
            self.gdp / 10.0,                                             # нормировка ~[−1..1]
            self.cpi / 20.0,                                             # ~[0..1]
            self.unemp / 15.0,                                           # ~[0..1]
            self.trust / 100.0,                                          # [0..1]
            self.gini / 60.0,                                            # ~[0.3..1]
            self.mobility,                                               # [0..1]
            self.wealth,                                                 # [0..1]
            self.employment,                                             # [0..1]
            self.tau_avg / 100.0,                                        # [0..1]
            self.crisis_intensity,                                       # [0..1]
            (self.year - 2000) / 50.0,                                   # нормировка года
            country_idx / max(N_COUNTRIES - 1, 1),                       # id страны [0..1]
        ], dtype=np.float32)


def _country_baseline(code: str, year: int) -> CountryState:
    """Сформировать начальное состояние страны для конкретного года."""
    idx_year = year - 2000                                               # индекс в макрорядах
    macro = MACRO[code]
    c_idx = COUNTRY_CODES.index(code)                                    # индекс страны
    # Базовые значения мобильности, благосостояния и занятости — из макроряда
    employment = max(0.0, 1.0 - macro["unemp"][idx_year] / 25.0)         # 1 − безраб./25
    # Благосостояние — обратное к Джини и пропорциональное ВВП
    wealth = 1.0 - macro["gini"][idx_year] / 70.0
    # Мобильность приближённо: чем ниже Джини и выше доверие, тем выше
    mobility = 0.5 * (1.0 - macro["gini"][idx_year] / 70.0) + \
               0.5 * (macro["trust"][idx_year] / 100.0)
    return CountryState(
        code=code, year=year,
        gdp=float(macro["gdp_growth"][idx_year]),
        cpi=float(macro["cpi"][idx_year]),
        unemp=float(macro["unemp"][idx_year]),
        trust=float(macro["trust"][idx_year]),
        gini=float(macro["gini"][idx_year]),
        mobility=float(mobility),
        wealth=float(wealth),
        employment=float(employment),
        tau_avg=float(macro["trust"][idx_year]),
        crisis_intensity=0.0,
        class_shares=CLASS_SHARES[c_idx].copy(),
    )


def _aggregate_crisis_intensity(code: str, year: int) -> Tuple[float, Dict[str, float]]:
    """Суммировать воздействия всех активных кризисов на страну в данном году."""
    total = {"gdp":0.0,"cpi":0.0,"unemp":0.0,"trust":0.0,"gini":0.0,"mobility":0.0}
    intensity = 0.0                                                      # суммарная сила
    for cr in ALL_CRISES:                                                # проход по каталогу
        if code not in cr.countries:                                     # не затронут?
            continue
        if cr.year_start <= year < cr.year_start + cr.duration:          # активен сейчас?
            # Затухание к концу: треугольный профиль (нарастание — пик — спад)
            mid = cr.year_start + cr.duration / 2.0
            half = max(cr.duration / 2.0, 0.5)
            decay = max(0.0, 1.0 - abs(year - mid) / half)               # 0..1
            w = cr.severity * decay
            for k in total:
                total[k] += cr.sign.get(k, 0.0) * w
            intensity = min(1.0, intensity + w)
    return intensity, total


class SocialEnvironment:
    """Многоагентная среда: 9 классов-агентов внутри одной страны.

    Шаг = 1 календарный год. Состояние страны эволюционирует по:
      1) Историческим макрорядам (если год ∈ 2000..2024) — как «учительский сигнал».
      2) Действиям агентов (вклад классов в макро через формулу 1 в статье).
      3) Экзогенным кризисам.
    Награда за шаг:
      - индивидуальная по агенту: классовый интерес ⟨β_class · ΔΩ⟩
      - системная: формула 3 (взвешенная по среднему классу)
    """

    def __init__(self, country_code: str, year_start: int, year_end: int,
                 hist_guidance: float = 0.6, noise: float = 0.02):
        self.country_code = country_code                                 # SE/DE/...
        self.country_idx = COUNTRY_CODES.index(country_code)
        self.year_start = year_start                                     # 2000 или 2026
        self.year_end = year_end                                         # 2024 или 2050
        self.hist_guidance = hist_guidance                               # сила «учительского сигнала»
        self.noise = noise                                               # стохастика среды
        self.state: CountryState
        self.t = 0
        self.history: List[CountryState] = []
        self.reset()

    def reset(self) -> np.ndarray:
        """Сбросить среду к начальному году."""
        self.t = 0                                                       # счётчик шагов
        self.state = _country_baseline(self.country_code, self.year_start)
        ci, _ = _aggregate_crisis_intensity(self.country_code, self.year_start)
        self.state.crisis_intensity = ci                                 # начальная интенсивность кризиса
        self.history = [self.state]                                      # лог состояний
        return self.state.as_state_vector(self.country_idx)

    def _apply_actions(self, actions: np.ndarray) -> np.ndarray:
        """Агрегировать действия 9 агентов взвешенно по долям классов (формула 1)."""
        # actions: (N_AGENTS, ACTION_DIM)  в [−1..1]
        w = self.state.class_shares / 100.0                              # веса классов
        agg = (w[:, None] * actions).sum(axis=0)                         # (ACTION_DIM,)
        return agg

    def step(self, actions: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float, bool, Dict[str, Any]]:
        """Один шаг среды = 1 год."""
        # 0. Предыдущее состояние сохраняем для расчёта дельт
        prev = self.state                                                # CountryState (копия по ссылке)
        prev_obs = np.array([prev.mobility, prev.wealth, prev.employment,
                              prev.tau_avg/100.0, prev.gini/60.0], dtype=np.float32)

        # 1. Следующий год
        next_year = prev.year + 1
        if next_year > self.year_end:                                    # эпизод заканчивается
            done = True
            return prev.as_state_vector(self.country_idx), \
                   np.zeros(N_AGENTS, dtype=np.float32), 0.0, done, \
                   {"reason": "year_end"}

        # 2. Кризис в новом году
        ci, eff = _aggregate_crisis_intensity(self.country_code, next_year)

        # 3. Исторический ориентир (если есть)
        if next_year <= 2024:                                            # обучение по реальным данным
            base = _country_baseline(self.country_code, next_year)
            target_obs = np.array([base.mobility, base.wealth, base.employment,
                                    base.tau_avg/100.0, base.gini/60.0], dtype=np.float32)
        else:                                                            # прогноз — без учителя
            base = None
            target_obs = None

        # 4. Действия агентов (форсаж со стороны общества)
        agg = self._apply_actions(actions)                               # (5,)

        # 5. Обновление макропараметров
        # ВВП: исторический ориентир + влияние совокупных действий + кризис + шум
        if base is not None:
            gdp_next = self.hist_guidance * base.gdp + \
                       (1.0 - self.hist_guidance) * (prev.gdp + 1.0*agg[1] - 0.5*agg[2])
        else:
            gdp_next = prev.gdp + 1.0*agg[1] - 0.5*agg[2]
        gdp_next += eff["gdp"]                                           # кризис
        gdp_next += np.random.normal(0.0, self.noise * 3.0)              # шум

        # Инфляция
        if base is not None:
            cpi_next = self.hist_guidance*base.cpi + (1.0-self.hist_guidance)*(prev.cpi + 0.3*agg[1])
        else:
            cpi_next = prev.cpi*0.7 + 0.3*max(0.0, prev.cpi + 0.5*agg[1])
        cpi_next += eff["cpi"]; cpi_next = max(-2.0, cpi_next)
        cpi_next += np.random.normal(0.0, self.noise * 2.0)

        # Безработица
        if base is not None:
            unemp_next = self.hist_guidance*base.unemp + (1.0-self.hist_guidance)*(prev.unemp - 0.5*agg[2])
        else:
            unemp_next = prev.unemp - 0.5*agg[2]
        unemp_next += eff["unemp"]; unemp_next = max(0.5, unemp_next)
        unemp_next += np.random.normal(0.0, self.noise * 2.0)

        # Доверие
        if base is not None:
            trust_next = self.hist_guidance*base.trust + (1.0-self.hist_guidance)*(prev.trust + 3.0*agg[3])
        else:
            trust_next = prev.trust + 3.0*agg[3]
        trust_next += 5.0*eff["trust"]; trust_next = float(np.clip(trust_next, 5.0, 95.0))

        # Джини
        if base is not None:
            gini_next = self.hist_guidance*base.gini + (1.0-self.hist_guidance)*(prev.gini - 0.5*agg[4])
        else:
            gini_next = prev.gini - 0.5*agg[4]
        gini_next += 1.0*eff["gini"]; gini_next = float(np.clip(gini_next, 20.0, 65.0))

        # Производные показатели
        employment = max(0.0, 1.0 - unemp_next/25.0)
        wealth = max(0.0, min(1.0, prev.wealth + 0.05*agg[1] - 0.02*eff["gini"]))
        mobility = max(0.0, min(1.0, prev.mobility + 0.05*agg[0] + 0.02*eff["mobility"]))
        tau_avg = trust_next

        # Доли классов: лёгкая динамика (мобильность переводит долю «вниз→вверх» среднего класса)
        shares = prev.class_shares.copy()
        m_drift = 0.10 * agg[0]                                          # сила восходящего смещения
        # Из прекариата/маргиналов вверх — в новый средний класс служащих
        delta = max(0.0, m_drift) * 0.3
        delta = min(delta, shares[7], shares[8])                         # не более, чем есть
        shares[7] -= delta * 0.5; shares[8] -= delta * 0.5
        shares[4] += delta * 0.6; shares[5] += delta * 0.4
        # Кризис увеличивает прекариат за счёт служащих и квалиф. рабочих
        c_drift = ci * 0.8
        shares[5] -= c_drift * 0.3; shares[4] -= c_drift * 0.2
        shares[7] += c_drift * 0.3; shares[8] += c_drift * 0.2
        shares = np.clip(shares, 0.2, None)
        shares = shares / shares.sum() * 100.0                           # нормировка

        # 6. Новое состояние
        self.state = CountryState(
            code=self.country_code, year=next_year,
            gdp=float(gdp_next), cpi=float(cpi_next),
            unemp=float(unemp_next), trust=float(trust_next), gini=float(gini_next),
            mobility=float(mobility), wealth=float(wealth), employment=float(employment),
            tau_avg=float(tau_avg), crisis_intensity=float(ci),
            class_shares=shares,
        )
        self.history.append(self.state)
        self.t += 1

        # 7. Дельты для функции награды
        new_obs = np.array([self.state.mobility, self.state.wealth, self.state.employment,
                             self.state.tau_avg/100.0, self.state.gini/60.0], dtype=np.float32)
        delta_obs = new_obs - prev_obs                                   # (Δm, Δw, Δe, Δτ, Δg)

        # 8. Индивидуальные награды (классовый интерес)
        agent_rewards = np.zeros(N_AGENTS, dtype=np.float32)
        for i in range(N_AGENTS):
            beta_i = CLASS_INTERESTS[i]                                  # (5,)
            # Для агентов: Джини с минусом (равенство — благо для всех, кроме элиты)
            sign_g = -1.0 if i > 0 else +0.3
            r = (beta_i[0]*delta_obs[0] + beta_i[1]*delta_obs[1] +
                 beta_i[2]*delta_obs[2] + beta_i[3]*delta_obs[3] +
                 sign_g * beta_i[4] * delta_obs[4])
            # Штраф за дисперсию действий (наказание за «шумные» решения)
            r -= RISK_AVERSION[i] * float(np.var(actions[i]))
            agent_rewards[i] = r

        # 9. Системная награда (формула 3) — взвешенная по среднему классу
        beta = BETA_SYSTEM
        sys_reward = (beta[0]*delta_obs[0] + beta[1]*delta_obs[1] +
                      beta[2]*delta_obs[2] + beta[3]*delta_obs[3] -
                      beta[4]*delta_obs[4])
        sys_reward += np.random.normal(0.0, self.noise)                  # ε

        info = {
            "crisis_intensity": ci,
            "crisis_effects": eff,
            "delta_obs": delta_obs,
            "agg_actions": agg,
            "target_obs": target_obs,                                    # для валидации (формула 2)
            "new_obs": new_obs,
        }
        done = next_year >= self.year_end
        return self.state.as_state_vector(self.country_idx), agent_rewards, float(sys_reward), done, info


# ======================================================================
#                 4. MADDPG: Actor, Critic, агенты и буфер
# ======================================================================

class Actor(nn.Module):
    """Политика агента π_i(s) → a_i. Малые сети — подходят для T4."""
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim), nn.Tanh(),                    # действия в [−1,1]
        )

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        return self.net(s)


class Critic(nn.Module):
    """Централизованный критик Q_i(s, a_1,…,a_N) — стандарт MADDPG."""
    def __init__(self, state_dim: int, n_agents: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + n_agents*action_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),                                        # скаляр-Q
        )

    def forward(self, s: torch.Tensor, a_all: torch.Tensor) -> torch.Tensor:
        x = torch.cat([s, a_all], dim=-1)
        return self.net(x)


class ReplayBuffer:
    """Кольцевой буфер для MADDPG."""
    def __init__(self, capacity: int, state_dim: int, n_agents: int, action_dim: int):
        self.capacity = capacity
        self.s  = np.zeros((capacity, state_dim), dtype=np.float32)
        self.a  = np.zeros((capacity, n_agents, action_dim), dtype=np.float32)
        self.r  = np.zeros((capacity, n_agents), dtype=np.float32)
        self.s2 = np.zeros((capacity, state_dim), dtype=np.float32)
        self.d  = np.zeros((capacity,), dtype=np.float32)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, d):
        i = self.idx
        self.s[i] = s; self.a[i] = a; self.r[i] = r; self.s2[i] = s2; self.d[i] = float(d)
        self.idx = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch: int):
        ix = np.random.randint(0, self.size, size=batch)
        return (torch.from_numpy(self.s[ix]).to(DEVICE),
                torch.from_numpy(self.a[ix]).to(DEVICE),
                torch.from_numpy(self.r[ix]).to(DEVICE),
                torch.from_numpy(self.s2[ix]).to(DEVICE),
                torch.from_numpy(self.d[ix]).to(DEVICE))


class MADDPG:
    """Полная многоагентная DDPG-схема с целевыми сетями."""
    def __init__(self, n_agents: int, state_dim: int, action_dim: int,
                 lr_a: float = 1e-4, lr_c: float = 1e-3, gamma: float = 0.95, tau: float = 0.01,
                 hidden_a: int = 128, hidden_c: int = 256):
        self.n = n_agents
        self.gamma = gamma
        self.tau = tau
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.actors  = [Actor(state_dim, action_dim, hidden_a).to(DEVICE) for _ in range(n_agents)]
        self.critics = [Critic(state_dim, n_agents, action_dim, hidden_c).to(DEVICE) for _ in range(n_agents)]
        self.actors_t  = [Actor(state_dim, action_dim, hidden_a).to(DEVICE) for _ in range(n_agents)]
        self.critics_t = [Critic(state_dim, n_agents, action_dim, hidden_c).to(DEVICE) for _ in range(n_agents)]
        for i in range(n_agents):
            self.actors_t[i].load_state_dict(self.actors[i].state_dict())
            self.critics_t[i].load_state_dict(self.critics[i].state_dict())
        self.opt_a = [Adam(self.actors[i].parameters(),  lr=lr_a) for i in range(n_agents)]
        self.opt_c = [Adam(self.critics[i].parameters(), lr=lr_c) for i in range(n_agents)]

    def act(self, state_vec: np.ndarray, noise_scale: float = 0.0) -> np.ndarray:
        """Получить действия всех агентов на текущем состоянии."""
        s = torch.from_numpy(state_vec).float().to(DEVICE).unsqueeze(0)
        acts = np.zeros((self.n, self.action_dim), dtype=np.float32)
        for i in range(self.n):
            with torch.no_grad():
                a = self.actors[i](s).cpu().numpy().squeeze(0)
            if noise_scale > 0:
                a = a + np.random.normal(0.0, noise_scale, size=a.shape).astype(np.float32)
            acts[i] = np.clip(a, -1.0, 1.0)
        return acts

    def _soft_update(self, target: nn.Module, source: nn.Module):
        for tp, sp in zip(target.parameters(), source.parameters()):
            tp.data.copy_(self.tau * sp.data + (1.0 - self.tau) * tp.data)

    def update(self, buffer: ReplayBuffer, batch: int) -> Dict[str, float]:
        """Один шаг обучения всех критиков и акторов."""
        if buffer.size < batch * 2:                                      # маленький буфер
            return {"c_loss": 0.0, "a_loss": 0.0}
        s, a, r, s2, d = buffer.sample(batch)

        # Действия целевых акторов на следующих состояниях
        with torch.no_grad():
            a2 = torch.stack([self.actors_t[i](s2) for i in range(self.n)], dim=1)  # (B,N,A)

        c_losses = []
        a_losses = []
        for i in range(self.n):
            # === Critic ===
            with torch.no_grad():
                a2_flat = a2.reshape(s2.shape[0], -1)
                q2 = self.critics_t[i](s2, a2_flat).squeeze(-1)
                y = r[:, i] + self.gamma * (1.0 - d) * q2
            a_flat = a.reshape(s.shape[0], -1)
            q = self.critics[i](s, a_flat).squeeze(-1)
            c_loss = F.mse_loss(q, y)
            self.opt_c[i].zero_grad(set_to_none=True)
            c_loss.backward()
            nn.utils.clip_grad_norm_(self.critics[i].parameters(), 1.0)
            self.opt_c[i].step()
            c_losses.append(float(c_loss.item()))

            # === Actor (DDPG-style) ===
            a_curr = a.clone()
            a_pred_i = self.actors[i](s)
            a_curr[:, i, :] = a_pred_i
            a_curr_flat = a_curr.reshape(s.shape[0], -1)
            a_loss = -self.critics[i](s, a_curr_flat).mean()
            self.opt_a[i].zero_grad(set_to_none=True)
            a_loss.backward()
            nn.utils.clip_grad_norm_(self.actors[i].parameters(), 1.0)
            self.opt_a[i].step()
            a_losses.append(float(a_loss.item()))

            # Мягкое обновление целевых сетей
            self._soft_update(self.actors_t[i],  self.actors[i])
            self._soft_update(self.critics_t[i], self.critics[i])

        return {"c_loss": float(np.mean(c_losses)),
                "a_loss": float(np.mean(a_losses))}


# ======================================================================
#                 5. Тренировка 2000–2025 и прогноз 2026–2050
# ======================================================================

# Гиперпараметры
EPISODES_TRAIN = 120                                                     # эпизодов на страну
EPISODES_WARMUP = 8                                                      # «прогрев» буфера (только сбор)
BATCH = 256                                                              # размер батча
NOISE_START = 0.30                                                       # начальный шум исследования
NOISE_END = 0.03                                                         # конечный шум
BUFFER_CAPACITY = 200_000                                                # ёмкость буфера
T_TRAIN = len(YEARS_HIST)                                                # 25 шагов (2000→2024)
T_FCAST = 25                                                             # 2026..2050


def validation_function(prediction: np.ndarray, target: np.ndarray,
                        mobility: float, gini: float) -> float:
    """Формула 2 из статьи: V_t = exp(−‖ŷ−y‖ · (1 + noise(m,g))).

    Высокое значение V_t → прогноз модели близок к историческим данным,
    с поправкой на «социальный шум» (мобильность + неравенство).
    """
    err = float(np.linalg.norm(prediction - target))                     # норма ошибки
    noise = 0.5 * (1.0 - mobility) + 0.5 * (gini / 60.0)                 # шум среды
    return float(np.exp(-err * (1.0 + noise)))


def train_country(code: str, log_each: int = 10) -> Dict[str, Any]:
    """Тренировка MADDPG на исторических данных одной страны."""
    print(f"\n[train] {code} ({COUNTRIES_RU[COUNTRY_CODES.index(code)]}) — "
          f"эпизодов {EPISODES_TRAIN}, шагов {T_TRAIN-1}, годы 2000–2024")
    env = SocialEnvironment(country_code=code, year_start=2000, year_end=2024,
                            hist_guidance=0.7)
    buffer = ReplayBuffer(BUFFER_CAPACITY, STATE_DIM, N_AGENTS, ACTION_DIM)
    maddpg = MADDPG(N_AGENTS, STATE_DIM, ACTION_DIM)

    ep_rewards: List[float] = []                                         # средняя системная награда за эп.
    ep_validation: List[float] = []                                      # среднее значение V_t
    losses_c: List[float] = []
    losses_a: List[float] = []

    pbar = tqdm(range(EPISODES_TRAIN), desc=f"[{code}] обучение", leave=True)
    for ep in pbar:
        s = env.reset()
        ep_sys = 0.0
        ep_val = []
        # Линейный спад шума
        noise_scale = NOISE_START + (NOISE_END - NOISE_START) * (ep / max(EPISODES_TRAIN-1,1))
        done = False
        while not done:
            actions = maddpg.act(s, noise_scale=noise_scale)
            s2, a_rew, sys_rew, done, info = env.step(actions)
            buffer.push(s, actions, a_rew, s2, done)
            s = s2
            ep_sys += sys_rew
            if info.get("target_obs") is not None:
                v = validation_function(info["new_obs"], info["target_obs"],
                                         env.state.mobility, env.state.gini)
                ep_val.append(v)

            if ep >= EPISODES_WARMUP:                                    # обучаемся после прогрева
                stats = maddpg.update(buffer, BATCH)
                losses_c.append(stats["c_loss"])
                losses_a.append(stats["a_loss"])

        ep_rewards.append(float(ep_sys))
        if ep_val:
            ep_validation.append(float(np.mean(ep_val)))
        if (ep + 1) % log_each == 0:
            pbar.set_postfix({
                "sys_R": f"{np.mean(ep_rewards[-log_each:]):+.3f}",
                "V": f"{np.mean(ep_validation[-log_each:]):.3f}" if ep_validation else "—",
                "noise": f"{noise_scale:.2f}",
            })
    print(f"[train] {code} → итог R={np.mean(ep_rewards[-10:]):+.3f}, "
          f"V={np.mean(ep_validation[-10:]):.3f}")
    return {
        "code": code,
        "maddpg": maddpg,
        "ep_rewards": ep_rewards,
        "ep_validation": ep_validation,
        "losses_c": losses_c,
        "losses_a": losses_a,
        "final_history": env.history,
    }


def forecast_country(code: str, maddpg: MADDPG, n_runs: int = 8) -> Dict[str, Any]:
    """Прогноз 2026–2050 на обученной политике (ансамбль из n_runs прогонов)."""
    print(f"[fcast] {code} → 2026–2050, ансамбль из {n_runs} прогонов")
    runs = []
    for r in range(n_runs):
        # Прогноз стартует с 2024 (последний год исторических данных) → 2050
        env = SocialEnvironment(country_code=code, year_start=2024, year_end=2050,
                                hist_guidance=0.0, noise=0.03)
        s = env.reset()
        done = False
        while not done:
            actions = maddpg.act(s, noise_scale=0.05)
            s, _, _, done, _ = env.step(actions)
        runs.append(env.history)
    # Усредняем по ансамблю
    years_f = list(range(2024, 2051))                                    # 2024..2050
    mean = {"gdp":[], "cpi":[], "unemp":[], "trust":[], "gini":[],
            "mobility":[], "wealth":[]}
    for t, yr in enumerate(years_f):
        for k in mean:
            v = [getattr(runs[r][t], k) for r in range(n_runs)]
            mean[k].append(float(np.mean(v)))
    return {"years": years_f, "mean": mean, "runs": runs}


# ======================================================================
#                 6. Таблицы и графики
# ======================================================================

def export_tables(results: Dict[str, Dict[str, Any]],
                  forecasts: Dict[str, Dict[str, Any]]) -> List[str]:
    """Сохранить ≥5 таблиц (.xlsx) с двуязычными заголовками."""
    paths = []

    # Т1. Доли классов по 6 странам
    df1 = pd.DataFrame(CLASS_SHARES, index=COUNTRIES_RU,
                       columns=[f"{ru} / {en}" for ru, en in zip(CLASS_NAMES_RU, CLASS_NAMES_EN)])
    df1.index.name = "Страна / Country"
    p1 = os.path.join(TABDIR, "table1_class_shares.xlsx")
    df1.to_excel(p1)
    paths.append(p1)

    # Т2. Веса интересов классов (β_i)
    df2 = pd.DataFrame(CLASS_INTERESTS,
                       index=[f"{ru} / {en}" for ru, en in zip(CLASS_NAMES_RU, CLASS_NAMES_EN)],
                       columns=["β1: Мобильность / Mobility",
                                "β2: Благосостояние / Wealth",
                                "β3: Занятость / Employment",
                                "β4: Доверие / Trust",
                                "β5: Неравенство / Inequality"])
    df2["Неприятие риска / Risk aversion"] = RISK_AVERSION
    p2 = os.path.join(TABDIR, "table2_class_interests.xlsx")
    df2.to_excel(p2)
    paths.append(p2)

    # Т3. Каталог кризисов
    rows = []
    for cr in ALL_CRISES:
        rows.append({
            "Название (RU)": cr.name_ru,
            "Name (EN)": cr.name_en,
            "Год / Year": cr.year_start,
            "Длит. лет / Duration": cr.duration,
            "Тип / Type": cr.ctype,
            "Страны / Countries": ",".join(cr.countries),
            "Сила / Severity": cr.severity,
            "Источник / Source": cr.source,
        })
    df3 = pd.DataFrame(rows)
    p3 = os.path.join(TABDIR, "table3_crises_catalog.xlsx")
    df3.to_excel(p3, index=False)
    paths.append(p3)

    # Т4. Итоги обучения по странам (последние 10 эпизодов)
    rows = []
    for code, r in results.items():
        rows.append({
            "Код / Code": code,
            "Страна / Country": f"{COUNTRIES_RU[COUNTRY_CODES.index(code)]} / "
                                  f"{COUNTRIES_EN[COUNTRY_CODES.index(code)]}",
            "Сред. R за посл. 10 эп. / Mean R last 10 ep.":
                float(np.mean(r["ep_rewards"][-10:])) if r["ep_rewards"] else 0.0,
            "Сред. V за посл. 10 эп. / Mean V last 10 ep.":
                float(np.mean(r["ep_validation"][-10:])) if r["ep_validation"] else 0.0,
            "Сред. loss критика / Mean critic loss":
                float(np.mean(r["losses_c"][-500:])) if r["losses_c"] else 0.0,
            "Сред. loss актора / Mean actor loss":
                float(np.mean(r["losses_a"][-500:])) if r["losses_a"] else 0.0,
        })
    df4 = pd.DataFrame(rows)
    p4 = os.path.join(TABDIR, "table4_training_summary.xlsx")
    df4.to_excel(p4, index=False)
    paths.append(p4)

    # Т5. Сводный прогноз 2025–2050 по 6 странам (4 ключевых показателя)
    blocks = []
    for code in COUNTRY_CODES:
        f = forecasts[code]
        df = pd.DataFrame({
            "Год / Year": f["years"],
            f"{code} ВВП рост, % / GDP growth %": f["mean"]["gdp"],
            f"{code} Джини / Gini": f["mean"]["gini"],
            f"{code} Доверие, % / Trust %": f["mean"]["trust"],
            f"{code} Безработица, % / Unemployment %": f["mean"]["unemp"],
        })
        blocks.append(df.set_index("Год / Year"))
    df5 = pd.concat(blocks, axis=1).reset_index()
    p5 = os.path.join(TABDIR, "table5_forecast_2025_2050.xlsx")
    df5.to_excel(p5, index=False)
    paths.append(p5)

    # Т6. Эволюция долей классов 2025 → 2050 (последняя точка)
    rows = []
    for code in COUNTRY_CODES:
        f = forecasts[code]
        final_shares = f["runs"][0][-1].class_shares                     # из первого прогона
        baseline = CLASS_SHARES[COUNTRY_CODES.index(code)]
        for j in range(N_AGENTS):
            rows.append({
                "Код / Code": code,
                "Класс / Class": f"{CLASS_NAMES_RU[j]} / {CLASS_NAMES_EN[j]}",
                "2022, % / 2022, %": float(baseline[j]),
                "2050, % / 2050, %": float(final_shares[j]),
                "Δ, п.п. / Δ, p.p.": float(final_shares[j] - baseline[j]),
            })
    df6 = pd.DataFrame(rows)
    p6 = os.path.join(TABDIR, "table6_class_shifts_2022_2050.xlsx")
    df6.to_excel(p6, index=False)
    paths.append(p6)

    print(f"[export] таблицы: {len(paths)} файлов")
    return paths


def _save_fig(fig, name: str):
    fig.tight_layout()                                                   # убираем налезание заголовков
    p = os.path.join(FIGDIR, name + ".png")
    fig.savefig(p, dpi=300, bbox_inches="tight")
    p2 = os.path.join(FIGDIR, name + ".pdf")
    fig.savefig(p2, bbox_inches="tight")
    plt.close(fig)
    return p


def make_figures(results, forecasts) -> List[str]:
    """Сохранить ≥8 фигур в Ч/Б."""
    paths = []

    # F1. Кривые обучения R(ep) для всех стран
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, code in enumerate(COUNTRY_CODES):
        r = results[code]["ep_rewards"]
        ax.plot(r, color=GRAYSCALE[i], linestyle=LINESTYLES[i], linewidth=1.4,
                label=f"{COUNTRIES_RU[i]} / {COUNTRIES_EN[i]}")
    ax.set_xlabel("Эпизод / Episode")
    ax.set_ylabel("Кумулятивная награда системы R / System reward")
    ax.set_title("Рис. 1. Кривые обучения по странам / "
                 "Fig. 1. Training curves by country")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig1_training_curves"))

    # F2. Кривые V_t (валидация) по странам
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, code in enumerate(COUNTRY_CODES):
        v = results[code]["ep_validation"]
        ax.plot(v, color=GRAYSCALE[i], linestyle=LINESTYLES[i], linewidth=1.4,
                label=f"{COUNTRIES_RU[i]} / {COUNTRIES_EN[i]}")
    ax.set_xlabel("Эпизод / Episode")
    ax.set_ylabel("Валидация V_t (формула 2) / Validation V_t")
    ax.set_title("Рис. 2. Сходимость прогнозов модели к историческим данным / "
                 "Fig. 2. Model prediction convergence to historical data")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig2_validation_curves"))

    # F3. Прогноз ВВП 2025–2050 для 6 стран
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, code in enumerate(COUNTRY_CODES):
        f = forecasts[code]
        ax.plot(f["years"], f["mean"]["gdp"],
                color=GRAYSCALE[i], linestyle=LINESTYLES[i], marker=MARKERS[i],
                markersize=3, linewidth=1.4,
                label=f"{COUNTRIES_RU[i]} / {COUNTRIES_EN[i]}")
    ax.axhline(0, color="0.6", linewidth=0.6)
    ax.set_xlabel("Год / Year")
    ax.set_ylabel("ВВП, прирост % / GDP growth %")
    ax.set_title("Рис. 3. Прогноз темпов ВВП 2025–2050 / Fig. 3. GDP growth forecast")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig3_forecast_gdp"))

    # F4. Прогноз доверия 2025–2050
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, code in enumerate(COUNTRY_CODES):
        f = forecasts[code]
        ax.plot(f["years"], f["mean"]["trust"],
                color=GRAYSCALE[i], linestyle=LINESTYLES[i], marker=MARKERS[i],
                markersize=3, linewidth=1.4,
                label=f"{COUNTRIES_RU[i]} / {COUNTRIES_EN[i]}")
    ax.set_xlabel("Год / Year")
    ax.set_ylabel("Доверие правительству, % / Trust in government %")
    ax.set_title("Рис. 4. Прогноз доверия к институтам / Fig. 4. Institutional trust forecast")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig4_forecast_trust"))

    # F5. Прогноз Джини
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, code in enumerate(COUNTRY_CODES):
        f = forecasts[code]
        ax.plot(f["years"], f["mean"]["gini"],
                color=GRAYSCALE[i], linestyle=LINESTYLES[i], marker=MARKERS[i],
                markersize=3, linewidth=1.4,
                label=f"{COUNTRIES_RU[i]} / {COUNTRIES_EN[i]}")
    ax.set_xlabel("Год / Year")
    ax.set_ylabel("Коэф. Джини / Gini")
    ax.set_title("Рис. 5. Прогноз неравенства (Джини) / Fig. 5. Inequality forecast (Gini)")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig5_forecast_gini"))

    # F6. Доли классов 2022 vs 2050 — bar chart по странам (среднему классу)
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(N_COUNTRIES); width = 0.35
    mc_2022 = CLASS_SHARES[:, MIDDLE_CLASS_IDX].sum(axis=1)
    mc_2050 = np.array([forecasts[code]["runs"][0][-1].class_shares[MIDDLE_CLASS_IDX].sum()
                         for code in COUNTRY_CODES])
    ax.bar(x - width/2, mc_2022, width, color="0.30", hatch="",
           edgecolor="black", label="2022 (факт) / 2022 actual")
    ax.bar(x + width/2, mc_2050, width, color="0.85", hatch="///",
           edgecolor="black", label="2050 (прогноз) / 2050 forecast")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{ru}\n{en}" for ru, en in zip(COUNTRIES_RU, COUNTRIES_EN)],
                       fontsize=8)
    ax.set_ylabel("Доля среднего класса, % / Middle-class share %")
    ax.set_title("Рис. 6. Динамика среднего класса 2022 → 2050 / "
                 "Fig. 6. Middle-class dynamics 2022 → 2050")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig6_middle_class_shift"))

    # F7. Тепловая карта чувствительности классов к кризисам (среднее по странам)
    sensit = np.zeros((N_AGENTS, 6))                                     # 9 классов × 6 каналов
    channels = ["gdp", "cpi", "unemp", "trust", "gini", "mobility"]
    for ci, ch in enumerate(channels):
        for ai in range(N_AGENTS):
            # |coef| влияния канала на интерес агента
            mapping = {"gdp":1, "cpi":1, "unemp":2, "trust":3, "gini":4, "mobility":0}
            beta_idx = mapping[ch]
            sensit[ai, ci] = abs(CLASS_INTERESTS[ai, beta_idx])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(sensit, cmap="gray_r", aspect="auto")
    ax.set_yticks(range(N_AGENTS))
    ax.set_yticklabels([f"{ru}\n{en}" for ru, en in zip(CLASS_NAMES_RU, CLASS_NAMES_EN)],
                       fontsize=7)
    ax.set_xticks(range(len(channels)))
    ax.set_xticklabels(["ВВП/GDP","Инфл./CPI","Безраб./Unemp","Доверие/Trust",
                        "Джини/Gini","Мобиль./Mob"], fontsize=8, rotation=30, ha="right")
    for i in range(N_AGENTS):
        for j in range(len(channels)):
            ax.text(j, i, f"{sensit[i,j]:.1f}", ha="center", va="center",
                    color="black" if sensit[i,j]<1.0 else "white", fontsize=7)
    ax.set_title("Рис. 7. Чувствительность классов к макроканалам / "
                 "Fig. 7. Class sensitivity to macro channels")
    fig.colorbar(im, ax=ax, shrink=0.7, label="|β| (вес интереса) / interest weight")
    paths.append(_save_fig(fig, "fig7_sensitivity_heatmap"))

    # F8. Гистограмма частоты кризисов по типам
    types = {}
    for cr in ALL_CRISES:
        types[cr.ctype] = types.get(cr.ctype, 0) + 1
    items = sorted(types.items(), key=lambda x: -x[1])
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = [k for k, _ in items]; vals = [v for _, v in items]
    bars = ax.barh(range(len(items)), vals, color="0.50",
                   edgecolor="black", hatch="///")
    ax.set_yticks(range(len(items)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Число кризисов / Number of crises")
    ax.set_title(f"Рис. 8. Структура каталога кризисов (всего {len(ALL_CRISES)}) / "
                 f"Fig. 8. Crisis catalog structure (total {len(ALL_CRISES)})")
    for b, v in zip(bars, vals):
        ax.text(v+0.1, b.get_y()+b.get_height()/2, str(v), va="center", fontsize=8)
    paths.append(_save_fig(fig, "fig8_crisis_types"))

    # F9. Loss-кривые (актор + критик), США как репрезентативный пример
    fig, ax = plt.subplots(figsize=(7, 4.5))
    losses_c = results["US"]["losses_c"]; losses_a = results["US"]["losses_a"]
    # Сглаживание скользящим средним
    def smooth(x, w=200):
        if len(x) < w: return x
        c = np.convolve(x, np.ones(w)/w, mode="valid")
        return c
    ax.plot(smooth(losses_c), color="0.0", linestyle="-", linewidth=1.3,
            label="L_critic (MSE)")
    ax.plot(smooth([-a for a in losses_a]), color="0.45", linestyle="--",
            linewidth=1.3, label="−L_actor (Q-ожидание)")
    ax.set_xlabel("Шаг обновления / Update step")
    ax.set_ylabel("Сглаж. функция потерь / Smoothed loss")
    ax.set_title("Рис. 9. Сходимость MADDPG (пример: США) / "
                 "Fig. 9. MADDPG convergence (example: USA)")
    ax.legend(loc="best", fontsize=8)
    paths.append(_save_fig(fig, "fig9_loss_curves_us"))

    print(f"[export] графики: {len(paths)} файлов")
    return paths


# ======================================================================
#                 7. Главный конвейер
# ======================================================================

def main():
    t0 = time.time()
    print("="*80)
    print("Многоагентная модель социальной системы (MADDPG, 9 классов, 6 стран)")
    print("Период обучения: 2000–2024 | Прогноз: 2025–2050")
    print("="*80)

    results: Dict[str, Dict[str, Any]] = {}
    forecasts: Dict[str, Dict[str, Any]] = {}
    for code in COUNTRY_CODES:
        r = train_country(code)
        results[code] = r
        f = forecast_country(code, r["maddpg"], n_runs=6)
        forecasts[code] = f

    # Сохраним численную сводку
    summary = {
        "config": {
            "EPISODES_TRAIN": EPISODES_TRAIN, "BATCH": BATCH,
            "BUFFER_CAPACITY": BUFFER_CAPACITY,
            "N_AGENTS": N_AGENTS, "STATE_DIM": STATE_DIM, "ACTION_DIM": ACTION_DIM,
            "device": str(DEVICE),
            "seed": SEED, "T_TRAIN": T_TRAIN, "T_FCAST": T_FCAST,
            "n_crises": len(ALL_CRISES),
        },
        "countries": {
            code: {
                "name_ru": COUNTRIES_RU[COUNTRY_CODES.index(code)],
                "name_en": COUNTRIES_EN[COUNTRY_CODES.index(code)],
                "mean_R_last10": float(np.mean(results[code]["ep_rewards"][-10:])),
                "mean_V_last10": float(np.mean(results[code]["ep_validation"][-10:]))
                                 if results[code]["ep_validation"] else None,
                "final_class_shares_2050": forecasts[code]["runs"][0][-1].class_shares.tolist(),
                "forecast_mean": forecasts[code]["mean"],
                "forecast_years": forecasts[code]["years"],
            }
            for code in COUNTRY_CODES
        },
    }
    with open(os.path.join(LOGDIR, "summary_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # Экспорт
    table_paths = export_tables(results, forecasts)
    fig_paths = make_figures(results, forecasts)

    # Итоговая сводка
    elapsed = time.time() - t0
    print("\n" + "="*80)
    print(f"ГОТОВО за {elapsed/60:.1f} мин")
    print(f"  таблицы → {TABDIR} ({len(table_paths)} файлов)")
    print(f"  графики → {FIGDIR} ({len(fig_paths)} файлов)")
    print(f"  сводка  → {LOGDIR}/summary_metrics.json")
    print("="*80)
    # Краткие R/V по странам
    print("\nИтоговые метрики (последние 10 эпизодов):")
    for code in COUNTRY_CODES:
        rN = COUNTRIES_RU[COUNTRY_CODES.index(code)]
        print(f"  {code} {rN:>10s}:  R={np.mean(results[code]['ep_rewards'][-10:]):+.3f},  "
              f"V={np.mean(results[code]['ep_validation'][-10:]):.3f}")


if __name__ == "__main__":
    main()


[init] PyTorch 2.11.0+cu128  device=cuda  cuda=True
[init] GPU = Tesla T4, mem = 15.6 GB
[init] Всего кризисов в каталоге: 44 (исторических 24, перспективных 20)
Многоагентная модель социальной системы (MADDPG, 9 классов, 6 стран)
Период обучения: 2000–2024 | Прогноз: 2025–2050

[train] SE (Швеция) — эпизодов 120, шагов 24, годы 2000–2024


[SE] обучение:   0%|          | 0/120 [00:00<?, ?it/s]

[train] SE → итог R=+0.783, V=0.552
[fcast] SE → 2026–2050, ансамбль из 6 прогонов

[train] DE (Германия) — эпизодов 120, шагов 24, годы 2000–2024


[DE] обучение:   0%|          | 0/120 [00:00<?, ?it/s]

[train] DE → итог R=+0.735, V=0.710
[fcast] DE → 2026–2050, ансамбль из 6 прогонов

[train] RU (Россия) — эпизодов 120, шагов 24, годы 2000–2024


...

[train] BR → итог R=+2.356, V=0.390
[fcast] BR → 2026–2050, ансамбль из 6 прогонов

[train] KZ (Казахстан) — эпизодов 120, шагов 24, годы 2000–2024


[KZ] обучение:   0%|          | 0/120 [00:00<?, ?it/s]

[train] KZ → итог R=+1.800, V=0.674
[fcast] KZ → 2026–2050, ансамбль из 6 прогонов

[train] US (США) — эпизодов 120, шагов 24, годы 2000–2024


[US] обучение:   0%|          | 0/120 [00:00<?, ?it/s]

[train] US → итог R=+1.199, V=0.447
[fcast] US → 2026–2050, ансамбль из 6 прогонов
[export] таблицы: 6 файлов
[export] графики: 9 файлов

ГОТОВО за 14.9 мин
  таблицы → outputs/tables (6 файлов)
  графики → outputs/figures (9 файлов)
  сводка  → outputs/logs/summary_metrics.json

Итоговые метрики (последние 10 эпизодов):
  SE     Швеция:  R=+0.783,  V=0.552
  DE   Германия:  R=+0.735,  V=0.710
  RU     Россия:  R=+2.323,  V=0.534
  BR   Бразилия:  R=+2.356,  V=0.390
  KZ  Казахстан:  R=+1.800,  V=0.674
  US        США:  R=+1.199,  V=0.447


## Архивация и скачивание результатов

In [3]:
import shutil, os
shutil.make_archive('outputs_polis', 'zip', 'outputs')
print('Архив создан:', os.path.getsize('outputs_polis.zip')/1024, 'KB')
try:
    from google.colab import files
    files.download('outputs_polis.zip')
except Exception:
    print('Не в Colab — архив outputs_polis.zip в текущей папке')


Архив создан: 2598.4072265625 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>